# Telco Customer Churn — Exploratory Data Analysis

## What this notebook is for

First look at the raw Telco churn dataset before any of it is trusted downstream.
It has two jobs:

1. **Find out what needs cleaning.** `pipeline/ingest.py` writes into a typed
   PostgreSQL schema (`db/schema.sql`), so every column has to be checked against the
   type it is about to be inserted as. Anything that arrives as text where the schema
   says `BOOLEAN` or `DECIMAL` has to be caught here, not at the `INSERT`.
2. **Find out what is worth modelling.** Which columns look like they separate churners
   from non-churners, and how balanced the target is — that decides the metric
   `models/train.py` is scored on.

Analysis only. Nothing here writes to the database; the cleaning logic that comes out of
it lives in `pipeline/ingest.py`.

## The dataset

**IBM Telco Customer Churn** — `blastchar/telco-customer-churn` on Kaggle, pulled with
`kagglehub`. One row per customer of a fictional telecom company, **7,043 rows × 21
columns**, with a labelled churn outcome.

The columns fall into four groups:

| Group | Columns |
|---|---|
| Identity / demographics | `customerID`, `gender`, `SeniorCitizen`, `Partner`, `Dependents` |
| Account | `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod` |
| Services | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| Money + label | `MonthlyCharges`, `TotalCharges`, `Churn` |

`Churn` is the target: did this customer leave within the last month.

## Key questions

**On data quality — what has to be fixed before the load:**

- Which columns are actually binary, and are they all encoded the same way?
- Are the numeric columns really numeric, or is something arriving as text?
- Are there missing values, and are they missing in a way `.info()` will report?
- Are there duplicate rows?

**On the churn signal — what to model:**

- How imbalanced is `Churn`? (This decides whether accuracy is a usable metric.)
- Does churn vary by contract type, internet service, or payment method?
- Does `tenure` separate churners from non-churners?
- Are `tenure`, `MonthlyCharges`, and `TotalCharges` collinear — is `TotalCharges`
  just `tenure × MonthlyCharges` restated?

The first group is answered below. The second is started — the class balance is
settled — and the per-segment breakdowns are still to come.


In [35]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Dataset path:", path)
print("Files:", os.listdir(path))

Dataset path: C:\Users\cheth\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1
Files: ['WA_Fn-UseC_-Telco-Customer-Churn.csv']


### Download — one CSV, cached outside the repo

`kagglehub` resolves to a single file, `WA_Fn-UseC_-Telco-Customer-Churn.csv`, in a
local cache directory rather than in the project.

Two things worth noting:

- **The path is machine-specific** (`C:\Users\cheth\.cache\kagglehub\...`), so this cell
  is not reproducible on another machine as written. That is fine here — `data/raw/` is
  gitignored and the CSV is deliberately not committed — but anything that needs to run
  elsewhere should take the path from config rather than from the cache.
- **`kagglehub` is not in `requirements.txt`.** The notebook imports it, so a fresh
  environment following the README will fail on this cell. It needs adding.


In [36]:
csv_file = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(csv_file)

### Load — no dtypes given, so pandas guessed

`pd.read_csv` with no `dtype=` argument, which means every column type below is
**inferred from the file contents**, not declared. That inference is what the next few
cells are checking, and it is where the one real surprise in this dataset shows up.

No parse errors and no warnings — whatever is in `TotalCharges`, it was readable as
*something*.


In [37]:
for col in df.columns:
    unique_values = set(df[col].dropna().unique())

    if unique_values <= {"Yes", "No"} | {1, 0}:
        print(col)

SeniorCitizen
Partner
Dependents
PhoneService
PaperlessBilling
Churn


### Six binary columns — in two different encodings

The subset test `unique_values <= {"Yes", "No"} | {1, 0}` finds every column whose values
fit inside that combined set:

```
SeniorCitizen
Partner
Dependents
PhoneService
PaperlessBilling
Churn
```

**These six are not encoded the same way.** `SeniorCitizen` is `0`/`1` integers; the other
five are `"Yes"`/`"No"` strings. The scan catches both because the comparison set is the
union of the two vocabularies.

That split is the reason `clean_data()` in `pipeline/ingest.py` uses **two** conversions
rather than one:

```python
df["senior_citizen"] = df["senior_citizen"].astype(bool)          # already 0/1
df[col] = df[col].map({"Yes": True, "No": False})                  # Yes/No strings
```

Using `.astype(bool)` on a Yes/No column would return `True` for **every** non-empty
string — including `"No"` — which would silently destroy the `Churn` label. The scan
above is what establishes which columns go down which path.

Note what this scan does *not* catch: columns like `MultipleLines` and `OnlineSecurity`
are Yes/No-*ish* but carry a third value (`"No phone service"`, `"No internet service"`),
so they fall out of the subset test and stay as `VARCHAR` in the schema. That is
correct — collapsing them to boolean would throw away the distinction between "declined
the service" and "not eligible for it".


In [38]:
print(df.head())
print(df.info())
print(df["Churn"].value_counts())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

### The two findings that matter

**1. `TotalCharges` is text, not a number.**

```
 18   MonthlyCharges    7043 non-null   float64
 19   TotalCharges      7043 non-null   str
```

`MonthlyCharges` inferred as `float64`; `TotalCharges`, which holds the same kind of
value, did not. A column of numbers only reads as text if at least one entry is not a
number — here, blank strings for customers who have not been billed yet.

And `.info()` **cannot see them**: it reports `7043 non-null` because an empty string is
a perfectly valid string. A null check alone would pass this dataset and let the bad rows
through to a `DECIMAL(10,2)` column. This is exactly why `clean_data()` coerces first and
drops second:

```python
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])
```

`errors="coerce"` turns the unparseable blanks into real `NaN`, which `dropna` can then
remove. The ingest pipeline drops **11 rows** this way — 7,043 in, 7,032 out.

**2. The target is imbalanced.**

```
Churn
No     5174
Yes    1869
```

**26.5% churn, 73.5% retained.** The consequence for `models/train.py`: a model that
predicts "no churn" for every single customer scores **73.5% accuracy** while being
completely useless. Accuracy is not a usable metric on this dataset — training has to be
scored on **PR-AUC and recall on the churn class**, and the imbalance handled with
`class_weight="balanced"` or `scale_pos_weight`.

Also visible: `SeniorCitizen` is `int64` (the `0`/`1` encoding from the previous cell),
and `customerID` is a unique string key — the natural `PRIMARY KEY`, which is how
`db/schema.sql` declares it.


In [39]:
# Column names
print(df.columns)

# First 5 rows
print(df.head())

# Shape (rows, columns)
print(df.shape)

# Data types
print(df.dtypes)

# Summary info
print(df.info())

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...          

### Shape confirmed, and the dtype split in one view

`(7043, 21)`, and the dtype summary is the whole data-quality story in one line:

```
dtypes: float64(1), int64(2), str(18)
memory usage: 1.1 MB
```

**One** float column in a dataset with two money columns — `MonthlyCharges` is numeric and
`TotalCharges` is not, restated. The two `int64`s are `tenure` and `SeniorCitizen`. The
other 18 columns are text, most of them legitimately categorical.

At 1.1 MB the whole dataset fits comfortably in memory, so no chunking or sampling is
needed anywhere in this project.

*(This cell repeats `head()` and `info()` from the cell above it. Worth consolidating on a
cleanup pass — the duplicate output makes the notebook harder to read than it needs to be.)*

---

## Where this leaves us

**Answered — the data-quality questions:**

- Six binary columns, two encodings, handled by two separate casts in `clean_data()`
- `TotalCharges` is text hiding blank strings; coerce-then-drop removes 11 rows
- No true nulls, but `.info()` is not sufficient to prove that
- Target is 26.5% positive, so accuracy is out as a metric

**Still open — the churn-signal questions:**

- Churn rate by `Contract`, `InternetService`, `PaymentMethod`, and tenure bucket
- Distributions of `MonthlyCharges` and `TotalCharges` split by churn
- Correlation between `tenure`, `MonthlyCharges`, `TotalCharges` — expect
  `tenure × TotalCharges` to be strongly collinear, then decide whether to drop one
- Confirm the 11 blank-`TotalCharges` rows are all `tenure = 0`, which would explain them
  as genuinely new accounts rather than corrupt data
